# CardioSurv — Exploratory Data Analysis

**Author:** Tan Guan Han  
**Task:** Task 5 Part A — EDA notebook

Generates 5 publication-quality PNG figures that M6 will use in the report's Datasets section.

**Inputs:** `data/processed/cleaned.csv`  
**Outputs:** `reports/figures/eda/*.png`

## 1. Setup

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configure plot style
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"

# Create output directory
FIG_DIR = Path("../reports/figures/eda")
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete. Figures will be saved to:", FIG_DIR.resolve())

## 2. Load and Inspect Data

In [2]:
df = pd.read_csv("../data/processed/cleaned.csv")
print("Shape:", df.shape)
df.head()

In [3]:
df.info()

In [4]:
df.describe()

In [5]:
print("Null counts:")
print(df.isnull().sum())

## 3. Figure 1 — Class Balance

In [6]:
fig, ax = plt.subplots(figsize=(8, 5))

# Use RiskCategory, not HeartDisease
order = ["Low", "Medium", "High"]
counts = df["RiskCategory"].value_counts().reindex(order)

colors = ["#2ecc71", "#f39c12", "#e74c3c"]  # Low=green, Medium=orange, High=red

bars = ax.bar(order, counts.values, color=colors, edgecolor="black")
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f"{count}\n({100*count/len(df):.1f}%)",
            ha="center", va="bottom", fontsize=11)

ax.set_title("Class Balance — RiskCategory Distribution", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of patients")
ax.set_ylim(0, max(counts.values) * 1.15)

plt.tight_layout()
plt.savefig(FIG_DIR / "class_balance.png")
plt.show()
print("Saved: class_balance.png")

## 4. Figure 2 — Numeric Distributions

In [7]:
numeric_cols = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak"]
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, col in zip(axes, numeric_cols):
    ax.hist(df[col], bins=30, color="#3498db", edgecolor="black", alpha=0.8)
    ax.set_title(col, fontsize=12, fontweight="bold")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    ax.axvline(df[col].mean(), color="red", linestyle="--", linewidth=1.5, label=f"Mean: {df[col].mean():.1f}")
    ax.legend(fontsize=9)

plt.suptitle("Numeric Feature Distributions", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "numeric_distributions.png")
plt.show()
print("Saved: numeric_distributions.png")

## 5. Figure 3 — Categorical Breakdowns

In [8]:
cat_cols = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))

for ax, col in zip(axes, cat_cols):
    crosstab = pd.crosstab(df[col], df["HeartDisease"])
    crosstab.plot(kind="bar", stacked=True, ax=ax,
                  color=["#2ecc71", "#e74c3c"], edgecolor="black")
    ax.set_title(col, fontsize=12, fontweight="bold")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    ax.legend(["No Disease", "Disease"], fontsize=9)
    ax.tick_params(axis="x", rotation=0)

plt.suptitle("Categorical Features by Heart Disease Status", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "categorical_breakdowns.png")
plt.show()
print("Saved: categorical_breakdowns.png")

## 6. Figure 4 — Correlation Heatmap

In [9]:
numeric_df = df[numeric_cols + ["FastingBS", "HeartDisease"]].copy()
corr = numeric_df.corr(method="spearman")

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, vmin=-1, vmax=1, square=True,
            cbar_kws={"label": "Spearman correlation"}, ax=ax,
            linewidths=0.5, linecolor="white")
ax.set_title("Spearman Correlation Heatmap", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig(FIG_DIR / "correlation_heatmap.png")
plt.show()
print("Saved: correlation_heatmap.png")

## 7. Figure 5 — Age Distribution by Heart Disease Status

In [10]:
fig, ax = plt.subplots(figsize=(9, 6))

data_to_plot = [
    df[df["HeartDisease"] == 0]["Age"],
    df[df["HeartDisease"] == 1]["Age"],
]
bp = ax.boxplot(data_to_plot, tick_labels=["No Disease", "Disease"],
                patch_artist=True, widths=0.5)

for patch, color in zip(bp["boxes"], ["#2ecc71", "#e74c3c"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title("Age Distribution by Heart Disease Status", fontsize=13, fontweight="bold")
ax.set_ylabel("Age (years)")
ax.set_xlabel("Heart Disease")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / "source_comparison.png")
plt.show()
print("Saved: source_comparison.png")

## 8. Key Observations

**Observation 1 — Class Balance:**  
The RiskCategory distribution shows the 3-class target variable (Low, Medium, High risk levels).

**Observation 2 — Numeric Distributions:**  
Age is normally distributed centered around 54 years, indicating a middle-aged cohort. RestingBP and Cholesterol show right-skewed distributions with some high outliers consistent with severe cases.

**Observation 3 — Categorical Patterns:**  
Male patients dominate the disease group, consistent with established cardiovascular epidemiology. Asymptomatic chest pain (ASY) is the strongest disease indicator — patients with ASY are far more likely to have heart disease.

**Observation 4 — Correlations:**  
The Spearman heatmap shows Oldpeak has the strongest positive correlation with HeartDisease (~0.4), while MaxHR has a clear negative correlation (~-0.4). Age and FastingBS show weaker but meaningful relationships.

## Done

All 5 figures saved to `reports/figures/eda/`. The team can now use these in the report's Datasets section.